In [ ]:
import flopy.utils.binaryfile as bf
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Point to your Cell-by-Cell file (update the filename to match your model name)
cbc_file = os.path.join(run_dir, "gwf.cbc") 

# 2. Open the binary budget file (MODFLOW 6 outputs in double precision)
cbc_obj = bf.CellBudgetFile(cbc_file, precision='double')

# 3. Get the list of all simulation times saved in the file
times = cbc_obj.get_times()

# 4. Extract the WEL package data
# Note: The text parameter usually matches the package name. If 'WEL' returns empty, 
# you can use cbc_obj.get_unique_record_names() to find the exact string.
wel_data = cbc_obj.get_data(text='WEL') 

# 5. Loop through the data to calculate total inflow per time step
actual_qin = []
for step_data in wel_data:
    # step_data is a numpy recarray containing ['node', 'q', etc.]
    # We sum the 'q' (flux) for all cells in the well boundary for this time step
    total_flux = np.sum(step_data['q']) 
    actual_qin.append(total_flux)


tide_heads = []

# Tidal heads time-series
tides_ts = os.path.join(run_dir, 'ghb_ts.ts')

with open(tides_ts, 'r') as f:

    ts_line = None

    for line in f.readlines():
        if line.startswith('BEGIN timeseries'):
            ts_line = True
            continue
        if line.startswith('END timeseries'):
            ts_line = False
            break

        if not ts_line:
            continue
        else:
            parts = line.split()
            if len(parts) >= 2:
                time = float(parts[0])
                head = float(parts[1])
                tide_heads.append(head)

print("Extracted tidal heads:", np.array(tide_heads))


# 6. Plot the extracted fluxes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
ax1.plot(times, actual_qin, marker='o', linestyle='-', color='b', label='Simulated Q_in (from .cbc)')
# plt.plot(times, qghb, marker='s', linestyle='-', color='g', label='Simulated Q_ghb (from .cbc)')
    
ax2.plot(times, tide_heads, marker='s', linestyle='-', color='g', label='Tidal Heads (from .ts)')
# Optional: Overlay your intended analytical formula to verify a perfect match
# intended_qin = inflow * (1 + 0.5 * np.sin(2 * np.pi * np.array(times) / total_time))
# plt.plot(times, intended_qin, linestyle='--', color='r', label='Intended Forcing')

ax1.set_title("Validation of Time-Varying Inflow (WEL Package)")
ax1.set_xlabel("Simulation Time (Days)")
ax1.set_ylabel("Total Inflow Rate ($m^3/d$)")
ax1.grid(True)
ax1.legend()

ax2.set_title("Tidal Heads")
ax2.set_xlabel("Simulation Time (Days)")
ax2.set_ylabel("Head (m)")
ax2.grid(True)
ax2.legend()

plt.show()